# Combine dated outputs

This notebook loads every dated folder under `outputs/`, adds a `date` column to each CSV, and writes combined files to `outputs/combined/`.

In [13]:
from pathlib import Path

import pandas as pd
import yfinance as yf

base_dir = Path("outputs")
combined_dir = base_dir / "combined"
combined_dir.mkdir(exist_ok=True)
TOPIX_TICKERS = ["EWJ"]
TOPIX_OUTPUT_TICKER = "JAPAN_ETF"
TOPIX_NAME = "Japan Equity ETF (EWJ proxy)"

file_names = ["pnls.csv", "risk.csv", "marginal_risk.csv", "portfolio_breakdown.csv"]
date_dirs = sorted(path for path in base_dir.iterdir() if path.is_dir() and path.name.isdigit())

combined_data = {}
summary_rows = []
ticker_frames = []

for date_dir in date_dirs:
    portfolio_path = date_dir / "portfolio_breakdown.csv"
    if portfolio_path.exists():
        ticker_frame = pd.read_csv(portfolio_path)[["ticker", "company_name"]].drop_duplicates()
        ticker_frames.append(ticker_frame)

for file_name in file_names:
    frames = []
    for date_dir in date_dirs:
        csv_path = date_dir / file_name
        if not csv_path.exists():
            continue
        frame = pd.read_csv(csv_path)

        frame.insert(0, "date", date_dir.name)
        frames.append(frame)
        summary_rows.append({"date": date_dir.name, "file": file_name, "rows": len(frame)})

    combined_frame = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    output_name = file_name.replace(".csv", "_all_dates.csv")
    combined_frame.to_csv(combined_dir / output_name, index=False)
    combined_data[file_name] = combined_frame

pnls_all_dates = combined_data["pnls.csv"]
risk_all_dates = combined_data["risk.csv"]
marginal_risk_all_dates = combined_data["marginal_risk.csv"]
portfolio_breakdown_all_dates = combined_data["portfolio_breakdown.csv"]

ticker_lookup = pd.concat(ticker_frames, ignore_index=True).drop_duplicates(subset=["ticker"]).sort_values("ticker") if ticker_frames else pd.DataFrame(columns=["ticker", "company_name"])
ticker_lookup = pd.concat([
    ticker_lookup,
    pd.DataFrame([{"ticker": ticker, "company_name": TOPIX_NAME} for ticker in TOPIX_TICKERS])
], ignore_index=True).drop_duplicates(subset=["ticker"], keep="first").sort_values("ticker").reset_index(drop=True)
start_date = pd.to_datetime(date_dirs[0].name, format="%Y%m%d") if date_dirs else None
end_date = pd.to_datetime(date_dirs[-1].name, format="%Y%m%d") + pd.Timedelta(days=1) if date_dirs else None
stock_tickers = ticker_lookup["ticker"].tolist()

if stock_tickers and start_date is not None and end_date is not None:
    downloaded_prices = yf.download(
        tickers=stock_tickers,
        start=start_date,
        end=end_date,
        group_by="ticker",
        auto_adjust=False,
        progress=False
    )
    price_field = "Adj Close" if "Adj Close" in downloaded_prices.columns.get_level_values(1) else "Close"
    close_prices = downloaded_prices.xs(price_field, level=1, axis=1)
    stock_prices_all_dates = close_prices.reset_index()
    stock_prices_all_dates = stock_prices_all_dates.rename(columns={stock_prices_all_dates.columns[0]: "date"})
    stock_prices_all_dates = stock_prices_all_dates.melt(id_vars="date", var_name="ticker", value_name="price")
    stock_prices_all_dates = stock_prices_all_dates.dropna(subset=["price"])
    stock_prices_all_dates = stock_prices_all_dates.merge(ticker_lookup, on="ticker", how="left")
    stock_prices_all_dates["ticker"] = stock_prices_all_dates["ticker"].replace({ticker: TOPIX_OUTPUT_TICKER for ticker in TOPIX_TICKERS})
    stock_prices_all_dates["company_name"] = stock_prices_all_dates["company_name"].fillna(stock_prices_all_dates["ticker"])
    stock_prices_all_dates["date"] = pd.to_datetime(stock_prices_all_dates["date"]).dt.strftime("%Y%m%d")
    stock_prices_all_dates = stock_prices_all_dates[["date", "ticker", "company_name", "price"]]
    stock_prices_all_dates = stock_prices_all_dates.sort_values(["date", "ticker"]).drop_duplicates(subset=["date", "ticker"], keep="first").reset_index(drop=True)
else:
    stock_prices_all_dates = pd.DataFrame(columns=["date", "ticker", "company_name", "price"])

stock_prices_all_dates.to_csv(combined_dir / "stock_prices_all_dates.csv", index=False)
summary = pd.DataFrame(summary_rows).sort_values(["file", "date"]).reset_index(drop=True)

print(f"Loaded {len(date_dirs)} date folders")
print(f"Combined files written to: {combined_dir}")
display(summary)
display(pnls_all_dates.head())
display(risk_all_dates.head())
display(marginal_risk_all_dates.head())
display(portfolio_breakdown_all_dates.head())
display(stock_prices_all_dates.head())


Loaded 94 date folders
Combined files written to: outputs/combined


,date,file,rows
0,20251201,marginal_risk.csv,2
1,20251202,marginal_risk.csv,2
2,20251203,marginal_risk.csv,2
3,20251204,marginal_risk.csv,2
4,20251205,marginal_risk.csv,2
...,...,...,...
371,20260403,risk.csv,4
372,20260406,risk.csv,4
373,20260407,risk.csv,4
374,20260408,risk.csv,4


,date,sim_days,sim_index,pnl
0,20251201,1,1,-345.196551
1,20251201,1,2,650.780149
2,20251201,1,3,-444.834894
3,20251201,1,4,-617.733749
4,20251201,1,5,-18.150620


,date,scenario,sim_days,Portfolio Value,VaR_95,VaR_99,ES_95,ES_99
0,20251201,normal,1,31358.703125,-1476.640569,-2090.506692,-1853.911277,-2393.024802
1,20251201,stress,1,31358.703125,-2132.087283,-3032.763968,-2666.596783,-3402.514793
2,20251201,normal,10,31358.703125,-4383.861631,-6285.703865,-5583.812061,-7355.704187
3,20251201,stress,10,31358.703125,-6528.715087,-9116.203401,-8232.292216,-10716.077626
4,20251202,normal,1,31031.533203,-1412.884700,-1991.410808,-1766.118209,-2275.178300


,date,sim_days,excluded_ticker,VaR_95,VaR_99,ES_95,ES_99
0,20251201,1,8035.T,0.0,0.0,0.0,0.0
1,20251201,10,8035.T,0.0,0.0,0.0,0.0
2,20251202,1,8035.T,0.0,0.0,0.0,0.0
3,20251202,10,8035.T,0.0,0.0,0.0,0.0
4,20251203,1,8035.T,0.0,0.0,0.0,0.0


,date,ticker,company_name,quantity,price,market_value,weight
0,20251201,8035.T,Tokyo Electron,1,31358.703125,31358.703125,1.0
1,20251202,8035.T,Tokyo Electron,1,31031.533203,31031.533203,1.0
2,20251203,8035.T,Tokyo Electron,1,32498.837891,32498.837891,1.0
3,20251204,8035.T,Tokyo Electron,1,33529.917969,33529.917969,1.0
4,20251205,8035.T,Tokyo Electron,1,32855.750000,32855.750000,1.0


,date,ticker,company_name,price
0,20251201,4568.T,Daiichi Sankyo,3767.180420
1,20251201,7011.T,Mitsubishi Heavy Industries,3891.769531
2,20251201,8035.T,Tokyo Electron,31358.703125
3,20251201,JAPAN_ETF,Japan Equity ETF (EWJ proxy),79.553391
4,20251202,4568.T,Daiichi Sankyo,3664.564697
